In [129]:
from dotenv import load_dotenv
import os
load_dotenv()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
DATABASE_NAME = os.getenv("DATABASE_NAME")

In [130]:
# create embedding model

from langchain_huggingface import HuggingFaceEmbeddings


def get_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

embedding_model = get_embedding_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8913.74it/s]


In [131]:
# get vector size
def get_vector_size():
    return len(
        embedding_model.embed_query("Hello")
    )

vector_size = get_vector_size()

In [132]:
#create qdrant client

from qdrant_client import QdrantClient
import qdrant_client
def get_qdrant_client():
    return QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY
    )
    
client = get_qdrant_client()
client.get_collections()
# print(qdrant_client.__version__)

CollectionsResponse(collections=[CollectionDescription(name='enterprise_docs'), CollectionDescription(name='knowledge_base'), CollectionDescription(name='vector_db')])

In [134]:
#create collection if it doesn't exixts

from qdrant_client.models import VectorParams,Distance

def create_collection_if_not_exists():
    collections = client.get_collections().collections

    collections_name = [collection.name for collection in collections]

    if DATABASE_NAME not in collections_name:
        client.create_collection(
            collection_name=DATABASE_NAME,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            )
        )

        print("✅ Collection Created")

       

        print("✅ Payload Indexes Created")
    else:
        print("ℹ️ Collection Already Exists")

create_collection_if_not_exists()


ℹ️ Collection Already Exists


In [135]:
 
from qdrant_client.models import PayloadSchemaType
 # Create payload index for department
client.create_payload_index(
    collection_name=DATABASE_NAME,
    field_name="metadata.department",
    field_schema=PayloadSchemaType.KEYWORD
)

# Create payload index for source
client.create_payload_index(
    collection_name=DATABASE_NAME,
    field_name="metadata.source",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=11, status=<UpdateStatus.COMPLETED: 'completed'>)

In [136]:
# create vector store

from langchain_qdrant import QdrantVectorStore


def create_vector_store():
    return QdrantVectorStore(
        client=client,
        embedding=embedding_model,
        collection_name=DATABASE_NAME
    )

vector_store = create_vector_store()

In [137]:
# create ingestion pipeline

import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def ingest_document(pdf_path,department):
    # load pdf
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    chunks = text_splitter.split_documents(documents)

    # add  metadata
    for chunk in chunks:
        chunk.metadata.update({
            "source":os.path.basename(pdf_path),
            "department":department
        })
        

    # upload to qdrant
    vector_store.add_documents(chunks)

    print("=" * 50)
    print("📄 File:", os.path.basename(pdf_path))
    print("📑 Pages:", len(documents))
    print("Departmen:", department)
    print("✂️ Chunks:", len(chunks))
    print("✅ Uploaded Successfully")
    print("=" * 50)


In [138]:
ingest_document("data/Grammer.pdf",department="Language")
ingest_document("data/Insurnace.pdf",department="Insurance")
ingest_document("data/sample-return-refund-policy-template.pdf",department="Refund")

Ignoring wrong pointing object 5 0 (offset 0)


📄 File: Grammer.pdf
📑 Pages: 6
Departmen: Language
✂️ Chunks: 53
✅ Uploaded Successfully
📄 File: Insurnace.pdf
📑 Pages: 5
Departmen: Insurance
✂️ Chunks: 23
✅ Uploaded Successfully
📄 File: sample-return-refund-policy-template.pdf
📑 Pages: 3
Departmen: Refund
✂️ Chunks: 11
✅ Uploaded Successfully


In [139]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def search_documents(query, department=None, k=3):

    search_filter = None

    if department:
        search_filter = Filter(
            must=[
                FieldCondition(
                    key="metadata.department",
                    match=MatchValue(value=department)
                )
            ]
        )

    retrieved_docs = vector_store.similarity_search(
        query=query,
        k=k,
        filter=search_filter
    )

    return retrieved_docs

In [141]:
query = "Tell me the details of Cashless Medical Coverage"
# query = "Tell me the dependency based rules"

results = search_documents(query=query,department="Insurance")

# print(f"Results: {results}")

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("=" *80)


for Final Approval.  After scrutinizing the 
document Cashless team will approved the 
Final Amount as per Bill given.  
Now Patient can pay only the Non 
admissible Item as mention in the Final Bill. 
After receiving the physical document from Hospital 
Medi Assist will Pay the Claim Amount Directly to 
Hospital Authority.  
Obtain  Referral/Authorization slip- 
issued by health center
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2025-04-18T07:55:30+00:00', 'source': 'Insurnace.pdf', 'total_pages': 5, 'page': 2, 'page_label': '3', 'department': 'Insurance', '_id': 'f0de2240-1e56-4bb1-9924-25e2c6b0ac17', '_collection_name': 'vector_db'}
for Final Approval.  After scrutinizing the 
document Cashless team will approved the 
Final Amount as per Bill given.  
Now Patient can pay only the Non 
admissible Item as mention in the Final Bill. 
After receiving the physical document from Hospital 
Medi Assist will Pay the Claim Amount Directly to 
Hospital Authorit